# M03.03 — When the corpus disagrees with itself

1. **Contradiction:** general terms against special terms.
2. **Two fixes**, and a related problem: one edition against another.
3. **Synthesis:** an answer spread over many articles.

## Setup

The shared `agentic_workshop.rag` module provides the document loader, structured splitter and retrieval helpers. The vector store's `query` takes an optional `keep` filter on metadata.

In [ ]:
import re

from agentic_workshop.rag import answer, build_store, embed, load_documents, show, split_structured

from agentic_workshop.client import get_client
from agentic_workshop.config import CHAT_MODEL
from agentic_workshop.documents import (
    GENERAL_CONDITIONS,
    POLICY_DOCUMENTS,
    editions_in_force,
)

client = get_client()
MODEL = CHAT_MODEL

### Document metadata

`POLICY_DOCUMENTS` (in `agentic_workshop/documents.py`) gives each document's product and effective date. We add whether it holds general or special terms.

In [ ]:
KIND = {
    doc_code: "general" if document["product"] == GENERAL_CONDITIONS else "special"
    for doc_code, document in POLICY_DOCUMENTS.items()
}

documents = load_documents()
pieces = [(name, p) for name, text in documents.items() for p in split_structured(text)]
store = build_store(documents, split_structured,
                    metadata=lambda doc_code: {**POLICY_DOCUMENTS[doc_code], "kind": KIND[doc_code]})

print(f"{len(store)} pieces from {len(documents)} documents")

## 1. One question, two answers

A Dentaire Plus policyholder has a cycling accident (`INV-2025-012`). How long do they have to report it?

In [ ]:
for name, text in documents.items():
    flat = " ".join(text.split())
    for m in re.finditer(r"[^.;]*(report accidents|Accidents must be declared)[^.;]*", flat):
        print(f"{name}: ...{m.group(0).strip()}...")

- `CAGA04-E2` Art. 31.3 (general conditions): 10 days.
- `DPAM01-E4` Art. 9.3 (special terms): 5 days.

Which one applies?

In [ ]:
print(next(p for n, p in pieces if n == "CAGA04-E2" and p.startswith("Art. 39")))

Special terms prevail, so the answer is 5 days. Ask the pipeline the question phrased two
ways.

In [ ]:
# SUGGESTION: If gpt-6-luna gets both phrasings right (or mentions both figures), pick phrasings where retrieval
# returns only one of the two articles; the points in the next cell rely on that.
questions = [
    "I have Dentaire Plus and broke a tooth in a cycling accident. "
    "Within how many days must I declare the accident to the insurer?",
    "I broke a tooth in a cycling accident and I have Dentaire Plus. "
    "How long do I have to report the accident?",
]

for q in questions:
    print(f"Q: {q}\n")
    show(store.query(embed([q])[0], k=3), preview=0)
    print(f"\n{answer(store, q, k=3)}\n")
    print("=" * 80, "\n")

- Whichever article makes the top 3 decides the answer.
- Neither answer mentions the other figure.

Where does Art. 39 rank?

In [ ]:
for q in questions:
    ranked = store.query(embed([q])[0], k=len(store))
    rank = next(i for i, r in enumerate(ranked, 1) if r["text"].startswith("Art. 39"))
    print(f"Art. 39 ranks {rank} of {len(ranked)}")

Near the bottom: it shares no vocabulary with the question.

## 2. Two fixes

### 2.1 Tell the model to report conflicts

In [ ]:
q = questions[0]
print(answer(store, q, k=5, instruction=(
    "The extracts may contradict each other. If they do, say so explicitly, give each "
    "conflicting figure with the document it came from, and state which you believe applies "
    "and why. Do not silently choose one."
)))

This helps when both figures reach the prompt, but the model still has to guess which document has authority. With the second phrasing, only one figure is retrieved at all.

### 2.2 Govern the context

Special terms are short. Put them in whole, retrieve only from the general conditions, label each extract, and state the precedence rule.

In [ ]:
def describe(m: dict) -> str:
    return f"{m['source']} · {m['kind']} terms · {m['product']} · effective {m['effective_date']}"


PRECEDENCE = (
    "Each extract is labelled as general or special terms. Where special terms and the general "
    "conditions differ, the special terms prevail (CGC Art. 39.2). Say which rule you applied."
)


def answer_governed(question: str, special_docs: set[str], k: int = 3) -> str:
    """Governing special terms in full, plus the top-k pieces of the general conditions."""
    special = [f"[{describe({'source': d, 'kind': KIND[d], **POLICY_DOCUMENTS[d]})} · full text]\n{documents[d]}"
               for d in sorted(special_docs)]
    general = store.query(embed([question])[0], k=k, keep=lambda m: m["kind"] == "general")
    context = "\n\n".join(special + [f"[{describe(r['metadata'])}]\n{r['text']}" for r in general])

    prompt = f"""Answer the question using only the extracts below.
{PRECEDENCE}

EXTRACTS
{context}

QUESTION
{question}
"""
    return client.responses.create(model=MODEL, input=prompt).output_text


for q in questions:
    print(f"Q: {q}\n\n{answer_governed(q, {'DPAM01-E4'})}\n")

Correct for both phrasings. The work moved into document governance: who holds which product, which terms are general or special, and the precedence rule.

### 2.3 Editions

`RFGM02-E3` (2024) and `RFBA02-E5` (2025) are both PrimaCare terms, and nearly identical.

In [ ]:
q_primacare = (
    "I was excluded from PrimaCare after two reminders for not seeing my GP first. "
    "When can I rejoin an insurance model with a limited choice of providers?"
)
show(store.query(embed([q_primacare])[0], k=6), preview=0)

Only the 2025 edition has a readmission rule (Art. 7). Select the edition in force on the date of the exclusion before retrieving, with `editions_in_force` from `agentic_workshop/documents.py`: for each product, the document with the latest effective date on or before that date.

In [ ]:
for date in ["2024-06-01", "2025-06-01"]:
    in_force = editions_in_force(date)
    print(f"--- excluded on {date}: searching {sorted(in_force)}\n")
    print(answer(store, q_primacare, k=3, label=describe,
                 keep=lambda m: m["source"] in in_force and m["product"] == "PrimaCare"))
    print()

Caveat: the header of `RFBA02-E5` names AMB Assurances SA, one of the insurers of the group. For a policyholder insured with another company of the group, this newer edition doesn't apply. "Latest effective date wins" is a business rule, not something the retriever can know.

## 3. A question no top-k can answer

In [ ]:
aggregate = (
    "What time limits, expressed in days, appear in the terms and conditions, "
    "and which one is the shortest?"
)

print(answer(store, aggregate, k=3))

Time limits appear in many articles across all six documents. Measure recall against `k`,
using a rough regex as the ground truth.

In [ ]:
DAYS = re.compile(r"\b(\d+|five|ten) days\b", re.IGNORECASE)

relevant = {i for i, text in enumerate(store.texts) if DAYS.search(" ".join(text.split()))}
print(f"{len(relevant)} of {len(store)} pieces mention a number of days, from "
      f"{len({store.metadata[i]['source'] for i in relevant})} documents\n")

query_vector = embed([aggregate])[0]
ranked = [r["index"] for r in store.query(query_vector, k=len(store))]

print(f"{'k':>4} {'recall':>8} {'found':>7} {'context chars':>14}")
for k in [1, 3, 5, 10, 20, 40, 80, len(store)]:
    found = set(ranked[:k]) & relevant
    chars = sum(len(store.texts[i]) for i in ranked[:k])
    print(f"{k:4d} {len(found) / len(relevant):8.0%} {len(found):3d}/{len(relevant):<3d} {chars:14d}")

- At `k=3` the full answer is out of reach, whatever the model.
- You cannot choose `k` in advance: how many pieces are relevant is only known once you have the
  answer.

In [ ]:
# With every document in context, the model has a chance.
full_corpus = "\n\n".join(f"[{name}]\n{text}" for name, text in documents.items())
print(client.responses.create(
    model=MODEL,
    input=f"Answer the question using only the documents below.\n\n{full_corpus}\n\nQUESTION\n{aggregate}",
).output_text)

In [ ]:
# And here is the shortest time limit computed from the corpus directly, with no model at all.
WORDS = {"five": 5, "ten": 10}
limits = [
    (int(WORDS.get(m.group(1).lower(), m.group(1))), store.metadata[i]["source"],
     store.texts[i].splitlines()[0])
    for i in relevant
    for m in DAYS.finditer(" ".join(store.texts[i].split()))
]
shortest = min(days for days, _, _ in limits)
print(f"shortest: {shortest} days")
for days, source, heading in sorted(set(limits)):
    if days == shortest:
        print(f"  {source} {heading}")

- The regex answers exactly and for free, but it is rough. It counts the 15 days in the war exclusion and misses "weeks" and "month".
- Options at scale: structured extraction (M02), map-reduce, long context, or agentic retrieval.

## Takeaways

- Similarity is not authority. The rule that resolves a conflict (here, Art. 39.2: special terms prevail) rarely shares vocabulary with the question, and ranks near the bottom.
- A contradiction in the corpus comes out as one confident answer, and which one depends on the phrasing.
- Asking the model to report conflicts helps only if both figures are retrieved. What works is putting the special terms in whole and stating the precedence rule: governance, not retrieval.
- Put authority in metadata and code, not in the retriever, and make someone responsible for maintaining it.
- Top-k cannot answer aggregation questions: you can't choose `k` in advance. Check whether a filter or a query can.

## Extensions

**Easy**

1. **Break the ground truth.** What does the `DAYS` regex wrongly include, and what does it miss (weeks, months)? Check how well embeddings handle negation ("not covered" against "covered").
2. **Measure the conflict reporting.** Run each prompt 20 times. How often does the answer give 5 days, 10 days, or both?

**Medium**

3. **Rank by authority.** Combine similarity with authority and recency scores. Test cases where the special terms are silent (infertility treatment under Optimum) and where they override the general conditions (scar correction).
4. **Map-reduce.** Extract `{obligation, days, article}` from each article, then aggregate in Python. Compare cost and latency with stuffing, at 6 documents and at 600.

**Hard**

5. **Detect conflicts before generating.** Compare retrieved extracts pairwise, or extract `(subject, value)` triples. Feed a conflict report into the prompt, or add an LLM re-ranker. When is a hand-written table of precedence rules simply better?